# Marketing Campaign Data Cleaning

## Imports & Dataset Load

In [166]:
# ==========================================================
# Objective:
# Identify and resolve common data quality issues in raw marketing campaign data to improve reporting reliability.
# ==========================================================

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

campaign_df = pd.read_csv(r'C:\Users\ARJUN\Python_projects\Python_Course\marketing_campaign_data_messy.csv')

print(f'The dataset has {campaign_df.shape[0]} and {campaign_df.shape[1]} columns.')

campaign_df.head(10)

The dataset has 2020 and 12 columns.


,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA
5,CMP-00006,Q4_BlackFriday_CMP-00006,2023-10-15 00:00:00,2023-10-28,Instagram,83386,2643,2697.03,NaN,1,NaN,IN
6,CMP-00007,Q3_Launch_CMP-00007,2023-10-07 00:00:00,2023-10-23,Facebook,38194,1135,1232.76,178.0,Yes,NaN,FA
7,CMP-00008,Q4_Launch_CMP-00008,2023-05-23,2023-05-28,Instagram,88498,1173,865.7,127.0,1,NaN,IN
8,CMP-00009,Q4_BlackFriday_CMP-00009,2023-03-23 00:00:00,2023-04-01,Google Ads,45131,1179,1046.18,104.0,1,NaN,GO
9,CMP-00010,Q2_Winter_CMP-00010,2023-03-21 00:00:00,2023-04-01,Email,61263,1153,1623.56,NaN,0,NaN,EM


## Initial Dataset Inspection

In [167]:
# Inspect structure and identify cleaning requirements

campaign_df.info()
campaign_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0    Campaign_ID   2020 non-null   object 
 1   Campaign_Name  2020 non-null   object 
 2   Start_Date     2020 non-null   object 
 3   End_Date       2020 non-null   object 
 4   Channel        1919 non-null   object 
 5   Impressions    2020 non-null   int64  
 6   Clicks         2020 non-null   int64  
 7   Spend          2020 non-null   object 
 8   Conversions    1820 non-null   float64
 9   Active         2020 non-null   object 
 10  Clicks         40 non-null     float64
 11  Campaign_Tag   2020 non-null   object 
dtypes: float64(2), int64(2), object(8)
memory usage: 189.5+ KB


,Impressions,Clicks,Conversions,Clicks
count,2020.000000,2020.000000,1820.000000,40.000000
mean,49839.896040,1500.744059,186.085714,54856.200000
std,28579.637473,1084.765654,160.129172,30552.773369
min,1055.000000,11.000000,0.000000,2508.000000
25%,25033.500000,650.750000,68.000000,30164.750000
50%,50097.500000,1245.000000,142.000000,57707.500000
75%,74784.250000,2185.250000,257.000000,81497.500000
max,99875.000000,4812.000000,943.000000,99483.000000


## Step 1: Standardize Column Headers

In [168]:
# ----------------------------------------------------------
# Step 1: Header Standardization
#
# Purpose:
# Ensure consistent column naming for easier reference during downstream analysis.
# ----------------------------------------------------------

print('Original Headers:')
print(campaign_df.columns)

campaign_df.columns = (
    campaign_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

print('\nCleaned Headers:')
print(campaign_df.columns)

Original Headers:
Index([' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel',
       'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks',
       'Campaign_Tag'],
      dtype='object')

Cleaned Headers:
Index(['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel',
       'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks',
       'campaign_tag'],
      dtype='object')


## Step 2: Clean Currency Values

In [169]:
# ----------------------------------------------------------
# Step 2: Currency Value Cleaning
#
# Problem:
# Spend values contain formatting inconsistencies (currency symbols, commas, special characters).
#
# Solution:
# Remove non-numeric characters and convert to float.
# ----------------------------------------------------------

currency_issue_mask = (
    campaign_df['spend']
    .astype(str)
    .str.contains('\$')
)

print('Dirty Spend Samples:')
print(campaign_df.loc[currency_issue_mask, ['campaign_id', 'spend']].head(3))

campaign_df['spend'] = (
    campaign_df['spend']
    .astype(str)
    .str.replace(r'[^\d.-]', '', regex = True)
)

campaign_df['spend'] = pd.to_numeric(campaign_df['spend'])

print('\nCleaned Spend Samples:')
print(campaign_df.loc[dirty_spend_mask, ['campaign_id', 'spend']].head(3))

Dirty Spend Samples:
   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22

Cleaned Spend Samples:
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


## Step 3: Correct Categorical Typos

In [170]:
# ----------------------------------------------------------
# Step 3: Channel Standardization
#
# Purpose:
# Resolve inconsistent categorical labels that could fragment grouped analysis.
# ----------------------------------------------------------

print('Unique Channel Values Before Cleaning:')
print(campaign_df['channel'].unique())

channel_cleaning_map = {
    'Tik_Tok'    : 'TikTok',
    'Facebok'    : 'Facebook',
    'E-mail'     : 'Email',
    'Insta_gram' : 'Instagram',
    'Gogle'      : 'Google',
    'N/A'        : np.nan
}

campaign_df['channel'] = (
    campaign_df['channel']
    .replace(channel_cleaning_map)
)

print('\nUnique Channel Values After Cleaning:')
print(campaign_df['channel'].unique())

Unique Channel Values Before Cleaning:
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']

Unique Channel Values After Cleaning:
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan 'Google']


## Step 4: Normalize Boolean Values

In [171]:
# ----------------------------------------------------------
# Step 4: Boolean Normalization
#
# Problem:
# Mixed boolean representations create inconsistency.
#
# Solution:
# Map all values to True / False.
# ----------------------------------------------------------

print('Original Active Values:')
print(campaign_df['active'].unique())

active_cleaning_map = {
    'Y'  : True,
    '0'  : False,
    'No' : False,
    'Yes': True,
    '1'  : True
}

campaign_df['active'] = (
    campaign_df['active']
    .map(active_cleaning_map)
    .fillna(False)
    .astype(bool)
)

print('\nCleaned Active Values:')
print(campaign_df['active'].unique())

# Note: .map() returns NaN for unmapped values, while .replace() preserves the original values.

Original Active Values:
['Y' '0' 'No' 'True' 'Yes' '1' 'False']

Cleaned Active Values:
[ True False]


## Step 5: Parse Date Fields

In [172]:
# ----------------------------------------------------------
# Step 5: Date Parsing
#
# Convert object-type date columns into datetime format.
# ----------------------------------------------------------

print('Before Conversion:')
print(campaign_df['start_date'].dtype)
print(campaign_df['end_date'].dtype)

campaign_df['start_date'] = pd.to_datetime(campaign_df['start_date'])
campaign_df['end_date'] = pd.to_datetime(campaign_df['end_date'])

print('\nAfter Conversion:')
print(campaign_df['start_date'].dtype)
print(campaign_df['end_date'].dtype)

Before Conversion:
object
object

After Conversion:
datetime64[ns]
datetime64[ns]


## Step 6: Logical Validation

In [173]:
# ----------------------------------------------------------
# Step 6: Logical Validation
#
# Business Rule: Clicks should never exceed impressions.
# ----------------------------------------------------------

campaign_df = (
    campaign_df.loc
    [:, ~ campaign_df.columns.duplicated()]
) # ~ is a bitwise operator that flips True to False and False to True

invalid_click_mask = (
    campaign_df['clicks'] > campaign_df['impressions']
)

print(f'Invalid click records found: {invalid_click_mask.sum()}')

Invalid click records found: 0


## Step 7: Correct Temporal Anomalies

In [174]:
# ----------------------------------------------------------
# Step 7: Temporal Integrity Check
#
# Business Rule: Campaign end date cannot precede start date.
# ----------------------------------------------------------

data_error_mask = campaign_df['end_date'] < campaign_df['start_date']

print('Temporal Errors:')
print(campaign_df.loc[data_error_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

campaign_df.loc[data_error_mask, 'end_date'] = (
    campaign_df
    .loc[data_error_mask, 'start_date'] 
    + 
    pd.Timedelta(days = 30)
)

print('\nAfter Correction:')
print(campaign_df.loc[data_error_mask, ['campaign_id', 'start_date', 'end_date']].head(3))

Temporal Errors:
   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-05-01
54   CMP-00055 2023-09-01 2023-08-27
71   CMP-00072 2023-02-01 2023-01-27

After Correction:
   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-06-05
54   CMP-00055 2023-09-01 2023-10-01
71   CMP-00072 2023-02-01 2023-03-03


## Step 8: Handle Outliers

In [175]:
# ----------------------------------------------------------
# Step 8: Outlier Treatment
#
# Method: IQR-based capping
#
# Objective:
# Reduce distortion caused by extreme spend values.
# ----------------------------------------------------------

Q1 = campaign_df['spend'].quantile(0.25)
Q3 = campaign_df['spend'].quantile(0.75)

IQR = Q3 - Q1 # range of the middle 50% of the data

upper_limit = Q3 + (3 * IQR)

spend_outlier_mask = campaign_df['spend'] > upper_limit

print('Outliers Before Capping:')
print(campaign_df.loc[spend_outlier_mask, ['campaign_id', 'spend']].head(3))

campaign_df.loc[spend_outlier_mask, 'spend'] = upper_limit

print('\nOutliers After Capping:')
print(campaign_df.loc[spend_outlier_mask, ['campaign_id', 'spend']].head(3))

Outliers Before Capping:
     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00

Outliers After Capping:
     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375


## Final Validation

In [176]:
print('Data Cleaning Completed Successfully\n')

campaign_df.info()
campaign_df.describe()

Data Cleaning Completed Successfully

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   campaign_id    2020 non-null   object        
 1   campaign_name  2020 non-null   object        
 2   start_date     2020 non-null   datetime64[ns]
 3   end_date       2020 non-null   datetime64[ns]
 4   channel        1919 non-null   object        
 5   impressions    2020 non-null   int64         
 6   clicks         2020 non-null   int64         
 7   spend          2020 non-null   float64       
 8   conversions    1820 non-null   float64       
 9   active         2020 non-null   bool          
 10  campaign_tag   2020 non-null   object        
dtypes: bool(1), datetime64[ns](2), float64(2), int64(2), object(4)
memory usage: 159.9+ KB


,impressions,clicks,spend,conversions
count,2020.000000,2020.000000,2020.000000,1820.000000
mean,49839.896040,1500.744059,1878.359681,186.085714
std,28579.637473,1084.765654,1643.700163,160.129172
min,1055.000000,11.000000,-2503.310000,0.000000
25%,25033.500000,650.750000,649.997500,68.000000
50%,50097.500000,1245.000000,1427.105000,142.000000
75%,74784.250000,2185.250000,2638.382500,257.000000
max,99875.000000,4812.000000,8603.537500,943.000000
